# 01 - Data Cleaning

**MRP: Machine Learning-Based Auto Insurance Fraud Detection**

Meshwa Patel (501390663) - Master of Data Science & Analytics

In [1]:
# Import the libraries
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## 1. Load the data and look at it

In [3]:
df = pd.read_csv("car_insurance_fraud_dataset.csv")
print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (30000, 24)


,policy_id,policy_state,policy_deductible,policy_annual_premium,insured_age,insured_sex,insured_education_level,insured_occupation,insured_hobbies,incident_date,incident_type,collision_type,incident_severity,authorities_contacted,incident_state,incident_city,incident_hour_of_the_day,number_of_vehicles_involved,bodily_injuries,witnesses,police_report_available,claim_amount,total_claim_amount,fraud_reported
0,POL100000,GA,400,1430.78,74,OTHER,High School,Manager,reading,6/13/2024,Parked Car,Front,Total Loss,NaN,MI,Charlesville,6,1,4,0,Yes,8161.36,11677.60,Y
1,POL100001,PA,300,854.49,74,MALE,College,Lawyer,chess,3/23/2025,Vehicle Theft,Rear,Total Loss,NaN,OH,Joshuaberg,0,3,4,5,No,18561.79,18027.81,N
2,POL100002,MI,400,1247.28,28,OTHER,PhD,Doctor,reading,1/26/2025,Parked Car,Unknown,Total Loss,Police,MI,Reynoldsfurt,14,4,4,1,No,10734.61,10375.59,N
3,POL100003,CA,600,622.42,37,MALE,PhD,Teacher,yachting,6/3/2024,Parked Car,Rear,Total Loss,Police,NC,Josephchester,22,3,3,5,No,13188.92,14204.34,N
4,POL100004,MI,700,1458.17,31,OTHER,PhD,Sales,reading,5/21/2024,Single Vehicle Collision,Side,Minor Damage,Fire,NY,Caitlinfort,18,4,2,4,No,21864.69,24038.84,N


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   policy_id                    30000 non-null  str    
 1   policy_state                 30000 non-null  str    
 2   policy_deductible            30000 non-null  int64  
 3   policy_annual_premium        30000 non-null  float64
 4   insured_age                  30000 non-null  int64  
 5   insured_sex                  30000 non-null  str    
 6   insured_education_level      30000 non-null  str    
 7   insured_occupation           30000 non-null  str    
 8   insured_hobbies              30000 non-null  str    
 9   incident_date                30000 non-null  str    
 10  incident_type                30000 non-null  str    
 11  collision_type               30000 non-null  str    
 12  incident_severity            30000 non-null  str    
 13  authorities_contacted      

## 2. Check missing values and duplicates

In [5]:
# Missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing)
print("\nMissing as % of rows:")
print((missing / len(df) * 100).round(2))

Columns with missing values:
authorities_contacted    7564
dtype: int64

Missing as % of rows:
authorities_contacted    25.21
dtype: float64


In [6]:
print("Number of duplicate rows:", df.duplicated().sum())

print("Unique policy_id values:", df["policy_id"].nunique(), "out of", len(df))

Number of duplicate rows: 0
Unique policy_id values: 30000 out of 30000


In [7]:
print(df["authorities_contacted"].value_counts(dropna=False))

authorities_contacted
Fire         7569
NaN          7564
Police       7498
Ambulance    7369
Name: count, dtype: int64


## 3. Handle missing values

In [8]:
df["authorities_contacted"] = df["authorities_contacted"].fillna("None")
print("Missing values left in the whole dataset:", df.isnull().sum().sum())
print(df["authorities_contacted"].value_counts())

Missing values left in the whole dataset: 0
authorities_contacted
Fire         7569
None         7564
Police       7498
Ambulance    7369
Name: count, dtype: int64


## 4. Fix data types and clean text columns

In [9]:
df["incident_date"] = pd.to_datetime(df["incident_date"], format="%m/%d/%Y")
print("Date range:", df["incident_date"].min().date(), "to", df["incident_date"].max().date())
print("Any dates that failed to parse:", df["incident_date"].isna().sum())

Date range: 2023-11-09 to 2025-11-08
Any dates that failed to parse: 0


In [10]:
# Strip stray spaces from all text columns
text_cols = df.select_dtypes(include="object").columns
for c in text_cols:
    df[c] = df[c].astype(str).str.strip()

# Quick sanity check on columns
for c in ["policy_state", "insured_sex", "insured_education_level",
          "insured_occupation", "insured_hobbies", "incident_type",
          "collision_type", "incident_severity", "authorities_contacted",
          "incident_state", "police_report_available"]:
           print(f"{c}: {df[c].nunique()} unique values")

policy_state: 10 unique values
insured_sex: 3 unique values
insured_education_level: 4 unique values
insured_occupation: 8 unique values
insured_hobbies: 7 unique values
incident_type: 4 unique values
collision_type: 4 unique values
incident_severity: 3 unique values
authorities_contacted: 4 unique values
incident_state: 10 unique values
police_report_available: 2 unique values


C:\Users\meshw\AppData\Local\Temp\ipykernel_13228\939978448.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns


## 5. Basic range checks (do the numbers make sense?)

In [11]:
num_cols = ["policy_deductible", "policy_annual_premium", "insured_age",
            "incident_hour_of_the_day", "number_of_vehicles_involved",
            "bodily_injuries", "witnesses", "claim_amount", "total_claim_amount"]
df[num_cols].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
policy_deductible,30000.0,562.78,250.41,200.00,300.00,600.00,800.00,1000.00
policy_annual_premium,30000.0,1051.26,260.36,600.02,826.32,1051.16,1277.90,1499.98
insured_age,30000.0,46.50,16.71,18.00,32.00,46.00,61.00,75.00
incident_hour_of_the_day,30000.0,11.50,6.90,0.00,6.00,12.00,17.00,23.00
number_of_vehicles_involved,30000.0,2.51,1.12,1.00,2.00,3.00,4.00,4.00
bodily_injuries,30000.0,2.00,1.41,0.00,1.00,2.00,3.00,4.00
witnesses,30000.0,2.50,1.71,0.00,1.00,2.00,4.00,5.00
claim_amount,30000.0,10823.08,6629.69,266.74,5350.95,10204.17,15381.87,29719.87
total_claim_amount,30000.0,12757.74,7028.92,502.18,6685.51,12740.43,18809.91,24999.72


## 6. Create the binary target

In [12]:
# fraud_reported is "Y"/"N".
df["fraud"] = (df["fraud_reported"] == "Y").astype(int)

print(df["fraud_reported"].value_counts())

print("\nFraud rate: {:.2f}%".format(100 * df["fraud"].mean()))

print("Class imbalance ratio (non-fraud : fraud) = {:.1f} : 1".format(
    (df["fraud"] == 0).sum() / (df["fraud"] == 1).sum()))

fraud_reported
N    26560
Y     3440
Name: count, dtype: int64

Fraud rate: 11.47%
Class imbalance ratio (non-fraud : fraud) = 7.7 : 1


## 7. Save the cleaned dataset

In [22]:
clean_path =  "car_insurance_clean.csv"
df.to_csv(clean_path, index=False)
print("Saved cleaned data to:", clean_path)
print("Final shape:", df.shape)

Saved cleaned data to: car_insurance_clean.csv
Final shape: (30000, 25)
